# A Theorem Prover for First-Order Logic without Equality 

This notebook implements a resolution-based theorem prover. It utilizes the `recursive-set` package for structural equality and immutable collections.

In [1]:
import { parseFormula as parse }              from "./FOL-Parser";
import { normalize }                          from "./FOL-CNF";
import { unify, Term as UTerm, Substitution } from "./Unification";
import { Tuple, RecursiveSet as RSet, RecursiveMap, Value } from "recursive-set";

## Auxiliary Functions

We need functions to manipulate literals and variables safely without resorting to type casting. We rely on type guards to transition between generic `Value` elements from `recursive-set` and strict AST nodes.

In [2]:
function isTuple(x: Value): x is Tuple<Value> {
    return x instanceof Tuple;
}

function complement(l: Tuple<Value>): Tuple<Value> {
    const arr = l.toArray();
    if (arr.length === 2 && arr[0] === '¬') {
        const inner = arr[1];
        if (isTuple(inner)) {
            return inner;
        }
    }
    return new Tuple('¬', l);
}

1:40 - Type 'Value' does not satisfy the constraint 'Value[]'.
1:40 - Type 'string' is not assignable to type 'Value[]'.
5:30 - Type 'Value' does not satisfy the constraint 'Value[]'.
5:30 - Type 'string' is not assignable to type 'Value[]'.
5:45 - Type 'Value' does not satisfy the constraint 'Value[]'.
5:45 - Type 'string' is not assignable to type 'Value[]'.
6:19 - Property 'toArray' does not exist on type 'Tuple<Value>'.
13:5 - Type 'Tuple<[string, Tuple<Value>]>' is not assignable to type 'Tuple<Value>'.
13:5 - Type '[string, Tuple<Value>]' is not assignable to type 'Value'.
13:5 - Type '[string, Tuple<Value>]' is not assignable to type 'Structural'.


We extract variables recursively. In our TypeScript parser, variables are strictly uppercase.

In [ ]:
function collectVariablesLit(l: Value): RSet<string> {
    if (typeof l === 'string') {
        if (/^[A-Z][a-zA-Z0-9_]*$/.test(l)) {
            return new RSet(l);
        }
        return new RSet();
    }
    if (isTuple(l)) {
        return l.toArray().reduce((acc, val) => acc.union(collectVariablesLit(val)), new RSet<string>());
    }
    if (Array.isArray(l)) {
        return l.reduce((acc, val) => acc.union(collectVariablesLit(val)), new RSet<string>());
    }
    return new RSet();
}

function collectVariablesClause(c: RSet<Tuple<Value>>): RSet<string> {
    return Array.from(c).reduce((acc, lit) => acc.union(collectVariablesLit(lit)), new RSet<string>());
}

To avoid variable capture during resolution, we standardize renaming clauses to fresh uppercase variables.

In [ ]:
const ascii_uppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".split("");

function applySubstitutionLit(l: Value, sigma: Map<string, string>): Value {
    if (typeof l === 'string') {
        const mapped = sigma.get(l);
        return mapped !== undefined ? mapped : l;
    }
    if (isTuple(l)) {
        return new Tuple(...l.toArray().map(v => applySubstitutionLit(v, sigma)));
    }
    if (Array.isArray(l)) {
        return l.map(v => applySubstitutionLit(v, sigma));
    }
    return l;
}

function renameVariables(f: RSet<Tuple<Value>>, g: RSet<Tuple<Value>>): RSet<Tuple<Value>> {
    const oldVars = Array.from(collectVariablesClause(f));
    const gVars = collectVariablesClause(g);
    const freshVars = ascii_uppercase.filter(v => !gVars.has(v));

    const sigma = new Map<string, string>();
    oldVars.forEach((v, i) => {
        if (i < freshVars.length) {
            sigma.set(v, freshVars[i]);
        }
    });

    let newClause = new RSet<Tuple<Value>>();
    for (const lit of f) {
        const applied = applySubstitutionLit(lit, sigma);
        if (isTuple(applied)) {
            newClause = newClause.add(applied);
        }
    }
    return newClause;
}

## Type-Safe Unification Bridge

We need to translate between `recursive-set` Tuples and the strict AST arrays expected by `Unification.ts`, ensuring rigorous structural validity without casts.

In [ ]:
function buildUTerm(arr: (UTerm | null)[]): UTerm | null {
    if (arr.length === 0) return null;
    const head = arr[0];
    if (typeof head !== 'string') return null;
    const tail: UTerm[] = [];
    for (let i = 1; i < arr.length; i++) {
        const elem = arr[i];
        if (elem === null) return null;
        tail.push(elem);
    }
    const result: [string, ...UTerm[]] = [head, ...tail];
    return result;
}

function valueToUTermSafe(v: Value): UTerm | null {
    if (typeof v === 'string') return v;
    if (isTuple(v)) {
        const mapped = v.toArray().map(valueToUTermSafe);
        return buildUTerm(mapped);
    }
    return null;
}

function uTermToValue(t: UTerm): Value {
    if (typeof t === 'string') return t;
    const mapped = t.map(uTermToValue);
    return new Tuple(...mapped);
}

function applyMu(v: Value, mu: Substitution): Value {
    if (typeof v === 'string') {
        const mapped = mu.get(v);
        return mapped !== undefined ? uTermToValue(mapped) : v;
    }
    if (isTuple(v)) {
        return new Tuple(...v.toArray().map(child => applyMu(child, mu)));
    }
    return v;
}

## Resolution and Factorization Core

In [ ]:
function resolve(C1: RSet<Tuple<Value>>, C2: RSet<Tuple<Value>>): RSet<RSet<Tuple<Value>>> {
    const C2Renamed = renameVariables(C2, C1);
    let result = new RSet<RSet<Tuple<Value>>>();

    for (const L1 of C1) {
        for (const L2 of C2Renamed) {
            const compL2 = complement(L2);
            const t1 = valueToUTermSafe(L1);
            const t2 = valueToUTermSafe(compL2);
            
            if (t1 !== null && t2 !== null) {
                const mu = unify(t1, t2);
                if (mu !== null) {
                    const C1Rem = Array.from(C1).filter(l => !l.equals(L1));
                    const C2Rem = Array.from(C2Renamed).filter(l => !l.equals(L2));
                    const resolventArr = [...C1Rem, ...C2Rem];
                    const resolventApplied = resolventArr.map(l => applyMu(l, mu));
                    
                    // Filter to only retain valid Tuples
                    const validApplied = resolventApplied.filter(isTuple);
                    result = result.add(new RSet(...validApplied));
                }
            }
        }
    }
    return result;
}

function factorize(C: RSet<Tuple<Value>>): RSet<RSet<Tuple<Value>>> {
    let result = new RSet<RSet<Tuple<Value>>>();
    const arrC = Array.from(C);
    for (let i = 0; i < arrC.length; i++) {
        for (let j = i + 1; j < arrC.length; j++) {
            const L1 = arrC[i];
            const L2 = arrC[j];
            const t1 = valueToUTermSafe(L1);
            const t2 = valueToUTermSafe(L2);
            
            if (t1 !== null && t2 !== null) {
                const mu = unify(t1, t2);
                if (mu !== null) {
                    const applied = arrC.map(l => applyMu(l, mu));
                    const validApplied = applied.filter(isTuple);
                    result = result.add(new RSet(...validApplied));
                }
            }
        }
    }
    return result;
}

## Automated Prover & Saturation

In [ ]:
type Reason = [RSet<Tuple<Value>>] | [RSet<Tuple<Value>>, RSet<Tuple<Value>>];

function infere(Clauses: RSet<RSet<Tuple<Value>>>): Array<[RSet<Tuple<Value>>, Reason]> {
    const result: Array<[RSet<Tuple<Value>>, Reason]> = [];
    const arrClauses = Array.from(Clauses);

    for (const C1 of arrClauses) {
        for (const C2 of arrClauses) {
            const res = resolve(C1, C2);
            for (const C of res) {
                result.push([C, [C1, C2]]);
            }
        }
        const facts = factorize(C1);
        for (const C of facts) {
            result.push([C, [C1]]);
        }
    }
    return result;
}

function saturate(Cs: RSet<RSet<Tuple<Value>>>): RecursiveMap<RSet<Tuple<Value>>, Reason> {
    let Clauses = new RSet(...Array.from(Cs));
    let cnt = 1;
    // Leveraging RecursiveMap for deep structural key association
    const Reasons = new RecursiveMap<RSet<Tuple<Value>>, Reason>();
    const emptyClause = new RSet<Tuple<Value>>();

    while (!Clauses.has(emptyClause)) {
        let added = false;
        const inferences = infere(Clauses);
        for (const [C, R] of inferences) {
            if (!Clauses.has(C)) {
                Reasons.set(C, R);
                Clauses = Clauses.add(C);
                added = true;
            }
        }
        console.log(`cnt = ${cnt}, number of clauses: ${Clauses.size}`);
        if (!added) break;
        cnt += 1;
    }
    return Reasons;
}

## Proof Reconstruction

In [ ]:
function stringifyClause(C: RSet<Tuple<Value>>): string {
    if (C.size === 0) return "{}";
    const arr = Array.from(C).map(l => {
        return JSON.stringify(l.toArray()).replace(/"/g, '').replace(/,/g, ', ');
    });
    return `{${arr.join(', ')}}`;
}

function updateProof(P1: string[], P2: string[]): string[] {
    const result = [...P1];
    for (const line of P2) {
        if (!result.includes(line)) result.push(line);
    }
    return result;
}

function constructProof(clause: RSet<Tuple<Value>>, Reasons: RecursiveMap<RSet<Tuple<Value>>, Reason>): string[] {
    const reason = Reasons.get(clause);
    if (reason === undefined) {
        return [`Axiom:       ${stringifyClause(clause)}`];
    }
    if (reason.length === 1) {
        const [C] = reason;
        const Proof = constructProof(C, Reasons);
        Proof.push(`Factorization: ${stringifyClause(C)} \n⊢            ${stringifyClause(clause)}`);
        return Proof;
    }
    if (reason.length === 2) {
        const [C1, C2] = reason;
        const ProofC1 = constructProof(C1, Reasons);
        const ProofC2 = constructProof(C2, Reasons);
        const Proof = updateProof(ProofC1, ProofC2);
        Proof.push(`Resolution:  ${stringifyClause(C1)},\n             ${stringifyClause(C2)}  \n⊢            ${stringifyClause(clause)}`);
        return Proof;
    }
    return [];
}

## Testing with the Red Dragons

We adapt the syntax so variables strictly begin with an uppercase letter, and predicates/functions strictly begin with a lowercase letter to conform to the TypeScript Parser rules.

In [ ]:
const s1 = '∀X:(∀Y:(child(Y, X) → canFly(Y)) → happy(X))';
const s2 = '∀X:(red(X) → canFly(X))';
const s3 = '∀X:(red(X) → ∀Y:(child(Y, X) → red(Y)))';
const s4 = '¬∀X:(red(X) → happy(X))';

const c1 = Array.from(normalize(parse(s1)));
const c2 = Array.from(normalize(parse(s2)));
const c3 = Array.from(normalize(parse(s3)));
const c4 = Array.from(normalize(parse(s4)));

const Clauses = new RSet<RSet<Tuple<Value>>>(...c1, ...c2, ...c3, ...c4);

console.log("Saturating...");
const Reasons = saturate(Clauses);
const Proof = constructProof(new RSet<Tuple<Value>>(), Reasons);

Proof.forEach(line => console.log(line));